In [1]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
from pathlib import Path
import torch
import cv2
import os
import shutil
import sys

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
detect_model = YOLO("models/11Lv2.pt")
pose_model = YOLO("yolo11m-pose")

In [3]:
video_input_directory = Path("video/input")
video_output_directory = Path("video/output")

video_input_directory.mkdir(parents=True, exist_ok=True)
video_output_directory.mkdir(parents=True, exist_ok=True)

for item in video_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [4]:
for video_index, video in enumerate(video_input_directory.glob("*.mp4")):
    print(f"Video: {video}")

    detect_results = detect_model.track(
        device=0,
        source=video,
        tracker="botsort.yaml",
        persist=True,
        conf=0.5,
        stream=True,
        verbose=False,
    )

    cap = cv2.VideoCapture(str(video))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1
    cap.release()
    
    output_video = str(video_output_directory / f"video{str(video_index)}.mp4")
    writer = cv2.VideoWriter(
        output_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    try:

        for result_index, result in enumerate(detect_results):
            
            print(f"\rProcessing Frame: {result_index}/{total_frames}", end="", flush=True)

            frame = result.orig_img.copy()
            boxes = result.boxes

            human_detected = False

            for box in boxes:
                cls = detect_model.names[int(box.cls[0])]
                if cls == "person":
                    human_detected = True
            
            if human_detected:
                pose_results = pose_model(
                    source=frame,
                    show_labels=False,
                    device=0,
                    conf=0.5,
                    verbose=False
                )
                frame = pose_results[0].plot(boxes=False)

            for box in boxes:

                xyxy = box.xyxy[0].cpu().numpy()
                conf = round(float(box.conf[0]), 2)
                cls = detect_model.names[int(box.cls[0])]
                track_id = int(box.id[0]) if box.id is not None else None

                x1, y1, x2, y2 = xyxy
                rect_rgb = (255,0,0)
                rect_thickness = 2
                cv2.rectangle(
                    frame,
                    (int(x1), int(y1)),
                    (int(x2), int(y2)),
                    rect_rgb,
                    rect_thickness
                )
                                
                font_scale = 1.0
                font_rgb = (255,0,0)
                font_thickness = 2
                label = f"ID:{track_id} CLS:{cls} CONF:{conf}"
                cv2.putText(
                    frame,
                    label,
                    (int(x1), int(y1)-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale,
                    font_rgb,
                    font_thickness
                )
            
            writer.write(frame)
    except Exception as e:
        print(f"\n\nCrash detected!")   
        printf("{%s}", e)
        if writer is not None:
            writer.release()
        sys.exit(1)
    
    print(f"\nCompleted {video}\n")
    writer.release()

print("All Done")


Video: video\input\clip2.mp4
Processing Frame: 3921/3921
Completed video\input\clip2.mp4

Video: video\input\clip4.mp4
Processing Frame: 693/693
Completed video\input\clip4.mp4

Video: video\input\clip5.mp4
Processing Frame: 2189/2189
Completed video\input\clip5.mp4

All Done
